# Les 05 - Agentic RAG


## Setup

Dit notitieboek toont het Agentic RAG (Retrieval-Augmented Generation) patroon met behulp van het Microsoft Agent Framework.

**Vereisten:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — jouw Azure AI Search service-eindpunt
- `AZURE_SEARCH_API_KEY` — jouw Azure AI Search API-sleutel
- Azure OpenAI implementatie geconfigureerd via omgevingsvariabelen
- Azure CLI geverifieerd (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Wat is Agentic RAG?

Traditionele RAG volgt een vaste pijplijn: documenten ophalen, vervolgens een antwoord genereren. **Agentic RAG** gaat verder door de agent autonomie te geven om te beslissen **wanneer** en **hoe** informatie wordt opgehaald.

Met Agentic RAG kan de agent:
- **Beslissen** of ophalen nodig is voordat een vraag beantwoord wordt
- **Kiezen** welke gegevensbron of tool geraadpleegd wordt
- **Evalueren** van opgehaalde resultaten en vervolg-ophalingen uitvoeren als de eerste poging onvoldoende is
- **Combineren** van informatie uit meerdere ophalingsstappen tot een samenhangend antwoord

Dit maakt de agent flexibeler en nauwkeuriger vergeleken met een statische ophalen-en-dan-genereren pijplijn.


## Een zoekhulpmiddel maken

In Agentic RAG worden externe gegevensbronnen verpakt als **tools** die de agent op aanvraag kan aanroepen. Hierdoor kan de agent ophalen behandelen als slechts een andere actie die hij kan uitvoeren, in plaats van een verplichte stap.

Hieronder definiëren we een reiskennisbank en stellen deze beschikbaar als een tool die de agent kan gebruiken om bestemmingsinformatie op te zoeken.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Het bouwen van de RAG-agent

Nu maken we een agent die de instructie krijgt om **altijd informatie op te halen voordat hij antwoord geeft**. De agent gebruikt het hulpmiddel `search_travel_knowledge` om zijn antwoorden te baseren op de kennisbank in plaats van te vertrouwen op zijn eigen trainingsgegevens.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratieve Ophaling — Het Maker-Checker Patroon

Een belangrijk voordeel van Agentic RAG is **iteratieve ophaling**. De agent kan meerdere zoekrondes uitvoeren om zijn aanvankelijke bevindingen te verifiëren, verfijnen of uit te breiden — vergelijkbaar met een "maker-checker" werkwijze:

1. **Makerstap**: De agent haalt initiële informatie op en stelt een conceptantwoord op.
2. **Checkerstap**: De agent voert extra zoekopdrachten uit om details te verifiëren of hiaten op te vullen.

Hieronder wordt de agent een vraag gesteld die meerdere bestemmingen vergelijkt, waardoor het meerdere keren moet zoeken.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Samenvatting

In deze les heb je geleerd hoe je een **Agentic RAG**-systeem bouwt met behulp van het Microsoft Agent Framework:

- **Agentic RAG** stelt agenten in staat om autonoom te beslissen wanneer ze informatie ophalen, waardoor ophalen dynamisch wordt in plaats van vast.
- **Hulpmiddelen als databronnen**: Externe kennisbanken (zoals Azure AI Search) worden verpakt als hulpmiddelen die de agent kan aanroepen.
- **Iteratief ophalen**: Het maker-checker patroon stelt de agent in staat om meerdere rondes van ophalen uit te voeren — zoeken, verifiëren en verfijnen — voordat een definitief antwoord wordt gegeven.

In productie zou je de in-memory `TRAVEL_KNOWLEDGE_BASE` vervangen door een echte Azure AI Search-index om grootschalige reisdocumentophalingen te verwerken.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
